### Mapper algorithm applied to voluntary terminations

#### 1. Import dependencies

In [ ]:
import snowflake.connector
import os
import matplotlib.pyplot as plt
import toml

from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN

from sklearn.datasets import make_circles
from tdamapper.learn import MapperAlgorithm
from tdamapper.cover import CubicalCover, BallCover
from tdamapper.plot import MapperPlot
from pathlib import Path

pwd = Path(os.path.abspath(''))
config = toml.load(pwd/'config.toml')
print(toml.dumps(config))


#### 2. Gather dataset

In [ ]:
connection = snowflake.connector.connect(**config['snowflake'])
cursor = connection.cursor()

val = cursor.execute(config['queries']['VT']).fetch_pandas_all()
print(val)


In [ ]:

# Generate toy dataset
X, labels = make_circles(n_samples=5000, noise=0.05, factor=0.3)
plt.figure(figsize=(5, 5))
plt.scatter(X[:,0], X[:,1], c=labels, s=.1, cmap="jet")
plt.axis("off")
plt.show()

# Apply PCA as lens
y = PCA(2).fit_transform(X)

# Mapper pipeline
cover = BallCover(radius=.05)
clust = DBSCAN()
graph = MapperAlgorithm(cover, clust).fit_transform(X, y)

# Visualize the Mapper graph
fig = MapperPlot(graph, dim=3, seed=42, iterations=200, layout_engine='networkx').plot_plotly(colors=labels)
fig.show()

In [ ]:
X